# Intermediate 03 — Token Exchange, Delegation & Impersonation

## Scenario

Alice delegates travel authority to a Travel Supervisor Agent. The supervisor may delegate only flight-search authority to a Flight Specialist Agent.

We build a mini Security Token Service (STS) that preserves:

- subject;
- current actor;
- actor history;
- task;
- audience;
- scope;
- expiry;
- delegation family;

while rejecting privilege amplification.


In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Optional
import json, time, uuid
import jwt

ISSUER = "https://sts.example"
SIGNING_KEY = "training-only-secret"

def now():
    return datetime.now(timezone.utc)

def ts(dt):
    return int(dt.timestamp())


## 1 — Issue subject and actor tokens

In [ ]:
def issue_token(*, subject, audience, scopes, ttl=600, extra=None):
    t = now()
    claims = {
        "iss": ISSUER,
        "sub": subject,
        "aud": audience,
        "scope": " ".join(sorted(scopes)),
        "iat": ts(t),
        "exp": ts(t + timedelta(seconds=ttl)),
        "jti": str(uuid.uuid4()),
    }
    if extra:
        claims.update(extra)
    return jwt.encode(claims, SIGNING_KEY, algorithm="HS256")

subject_token = issue_token(
    subject="user:alice",
    audience="https://sts.example",
    scopes={"travel:read","travel:book","expenses:read"},
    ttl=1800,
    extra={"may_act":{"sub":"agent:travel-supervisor"}},
)

actor_token = issue_token(
    subject="agent:travel-supervisor",
    audience="https://sts.example",
    scopes={"travel:read","travel:book"},
    ttl=900,
)

print(jwt.decode(
    subject_token,
    SIGNING_KEY,
    algorithms=["HS256"],
    audience="https://sts.example"
))


## 2 — Validate exchange inputs

In [ ]:
def validate_input_token(token, expected_audience="https://sts.example"):
    return jwt.decode(
        token,
        SIGNING_KEY,
        algorithms=["HS256"],
        issuer=ISSUER,
        audience=expected_audience,
    )

subject_claims = validate_input_token(subject_token)
actor_claims = validate_input_token(actor_token)
print(subject_claims["sub"], actor_claims["sub"])


## 3 — Validate `may_act`

In [ ]:
def may_actor_act_for(subject_claims, actor_claims):
    allowed = subject_claims.get("may_act", {})
    return allowed.get("sub") == actor_claims.get("sub")

assert may_actor_act_for(subject_claims, actor_claims)
print("Actor may act for subject")


## 4 — Token exchange request

In [ ]:
@dataclass(frozen=True)
class ExchangeRequest:
    subject_token: str
    actor_token: Optional[str]
    audience: str
    requested_scopes: frozenset[str]
    task_id: str
    resource_id: str
    delegation_family: str
    mode: str = "delegation"

req = ExchangeRequest(
    subject_token=subject_token,
    actor_token=actor_token,
    audience="travel-api",
    requested_scopes=frozenset({"travel:book"}),
    task_id="trip:483",
    resource_id="trip:483",
    delegation_family="dlg:483",
)
req


## 5 — Authority attenuation

In [ ]:
ACTOR_MAX = {
    "agent:travel-supervisor": {"travel:read","travel:book"},
    "agent:flight-specialist": {"flights:search"},
}

TASK_SCOPES = {
    "trip:483": {"travel:read","travel:book","flights:search"},
}

def scope_set(claims):
    return set(claims.get("scope","").split())

def attenuated_scope(subject_claims, actor, task_id, requested):
    allowed = (
        scope_set(subject_claims)
        & ACTOR_MAX.get(actor, set())
        & TASK_SCOPES.get(task_id, set())
    )
    requested = set(requested)
    if not requested <= allowed:
        raise PermissionError(f"excess authority requested: {requested - allowed}")
    return requested

print(attenuated_scope(
    subject_claims,
    actor_claims["sub"],
    "trip:483",
    {"travel:book"},
))


## 6 — Lifetime attenuation

In [ ]:
def remaining_lifetime(claims):
    return claims["exp"] - int(time.time())

def output_ttl(subject_claims, actor_claims, task_ttl=600, policy_max=300):
    return max(1, min(
        remaining_lifetime(subject_claims),
        remaining_lifetime(actor_claims),
        task_ttl,
        policy_max,
    ))

print("derived TTL:", output_ttl(subject_claims, actor_claims))


## 7 — Mini STS

In [ ]:
REVOKED_FAMILIES = set()
EXCHANGE_AUDIT = []

def issue_exchange(req):
    subject = validate_input_token(req.subject_token)
    actor = validate_input_token(req.actor_token) if req.actor_token else None

    if req.delegation_family in REVOKED_FAMILIES:
        raise PermissionError("delegation family revoked")

    if req.mode == "delegation":
        if actor is None:
            raise PermissionError("actor token required")
        if not may_actor_act_for(subject, actor):
            raise PermissionError("actor not authorized by subject")

        scopes = attenuated_scope(
            subject, actor["sub"], req.task_id, req.requested_scopes
        )

        out = issue_token(
            subject=subject["sub"],
            audience=req.audience,
            scopes=scopes,
            ttl=output_ttl(subject, actor),
            extra={
                "act":{"sub":actor["sub"]},
                "task_id":req.task_id,
                "resource_id":req.resource_id,
                "delegation_family":req.delegation_family,
            },
        )

    elif req.mode == "impersonation":
        scopes = set(req.requested_scopes)
        if not scopes <= scope_set(subject):
            raise PermissionError("scope exceeds subject")
        out = issue_token(
            subject=subject["sub"],
            audience=req.audience,
            scopes=scopes,
            ttl=min(120, remaining_lifetime(subject)),
            extra={
                "task_id":req.task_id,
                "resource_id":req.resource_id,
                "delegation_family":req.delegation_family,
            },
        )
    else:
        raise ValueError("unknown mode")

    claims = jwt.decode(
        out, SIGNING_KEY, algorithms=["HS256"],
        audience=req.audience, issuer=ISSUER
    )

    EXCHANGE_AUDIT.append({
        "exchange_id": str(uuid.uuid4()),
        "subject": claims["sub"],
        "actor": claims.get("act",{}).get("sub"),
        "audience": claims["aud"],
        "scope": claims["scope"],
        "task": claims["task_id"],
        "delegation_family": claims["delegation_family"],
        "issued_jti": claims["jti"],
    })
    return out

delegated = issue_exchange(req)
print(json.dumps(jwt.decode(
    delegated,
    SIGNING_KEY,
    algorithms=["HS256"],
    audience="travel-api",
    issuer=ISSUER
), indent=2))


## 8 — Delegation vs impersonation

In [ ]:
impersonation_req = ExchangeRequest(
    subject_token=subject_token,
    actor_token=None,
    audience="legacy-travel-api",
    requested_scopes=frozenset({"travel:read"}),
    task_id="trip:483",
    resource_id="trip:483",
    delegation_family="dlg:legacy",
    mode="impersonation",
)

impersonated = issue_exchange(impersonation_req)

delegated_claims = jwt.decode(
    delegated, SIGNING_KEY, algorithms=["HS256"],
    audience="travel-api", issuer=ISSUER
)
imp_claims = jwt.decode(
    impersonated, SIGNING_KEY, algorithms=["HS256"],
    audience="legacy-travel-api", issuer=ISSUER
)

print("delegated actor:", delegated_claims.get("act"))
print("impersonated actor:", imp_claims.get("act"))


## 9 — Multi-hop delegation

In [ ]:
next_subject_token = issue_token(
    subject="user:alice",
    audience="https://sts.example",
    scopes={"travel:read","travel:book","flights:search"},
    ttl=300,
    extra={
        "act":{"sub":"agent:travel-supervisor"},
        "may_act":{"sub":"agent:flight-specialist"},
        "task_id":"trip:483",
        "delegation_family":"dlg:483",
    },
)

flight_actor_token = issue_token(
    subject="agent:flight-specialist",
    audience="https://sts.example",
    scopes={"flights:search"},
    ttl=240,
)


## 10 — Nested `act` chain

In [ ]:
def build_act_claim(current_actor, prior_act=None):
    result = {"sub": current_actor}
    if prior_act:
        result["act"] = prior_act
    return result

def issue_child_exchange(subject_token, actor_token):
    subject = validate_input_token(subject_token)
    actor = validate_input_token(actor_token)

    if subject.get("may_act",{}).get("sub") != actor["sub"]:
        raise PermissionError("child actor not permitted")

    requested = {"flights:search"}
    allowed = (
        scope_set(subject)
        & ACTOR_MAX.get(actor["sub"], set())
        & TASK_SCOPES["trip:483"]
    )
    if not requested <= allowed:
        raise PermissionError("privilege amplification")

    return issue_token(
        subject=subject["sub"],
        audience="flight-api",
        scopes=requested,
        ttl=min(
            remaining_lifetime(subject),
            remaining_lifetime(actor),
            180
        ),
        extra={
            "act": build_act_claim(actor["sub"], subject.get("act")),
            "task_id":"trip:483",
            "resource_id":"trip:483",
            "delegation_family":"dlg:483",
        },
    )

child = issue_child_exchange(next_subject_token, flight_actor_token)

child_claims = jwt.decode(
    child, SIGNING_KEY, algorithms=["HS256"],
    audience="flight-api", issuer=ISSUER
)

print(json.dumps(child_claims, indent=2))


## 11 — Current actor and history

In [ ]:
def actor_chain(claims):
    node = claims.get("act")
    result = []
    while node:
        result.append(node["sub"])
        node = node.get("act")
    return result

chain = actor_chain(child_claims)
print("current actor:", chain[0])
print("history:", chain[1:])


## 12 — Historical actors do not confer current privilege

In [ ]:
ACTOR_MAX["agent:travel-supervisor"].add("admin")

current_actor = actor_chain(child_claims)[0]
print("current actor rights:", ACTOR_MAX[current_actor])
assert "admin" not in ACTOR_MAX[current_actor]


## 13 — Negative security tests

In [ ]:
def expect_denied(name, fn):
    try:
        fn()
        print(name, "FAIL")
        return False
    except Exception as e:
        print(name, "PASS ->", type(e).__name__)
        return True

assert expect_denied(
    "scope amplification",
    lambda: attenuated_scope(
        subject_claims,
        actor_claims["sub"],
        "trip:483",
        {"admin"},
    )
)

evil_actor = issue_token(
    subject="agent:evil",
    audience="https://sts.example",
    scopes={"travel:book"},
    ttl=200,
)

evil_req = ExchangeRequest(
    subject_token=subject_token,
    actor_token=evil_actor,
    audience="travel-api",
    requested_scopes=frozenset({"travel:book"}),
    task_id="trip:483",
    resource_id="trip:483",
    delegation_family="dlg:evil",
)

assert expect_denied("actor substitution", lambda: issue_exchange(evil_req))


## 14 — Delegation depth

In [ ]:
def delegation_depth(claims):
    return len(actor_chain(claims))

MAX_DEPTH = 2
print("depth:", delegation_depth(child_claims))
assert delegation_depth(child_claims) <= MAX_DEPTH


## 15 — Delegation family revocation

In [ ]:
def revoke_family(family):
    REVOKED_FAMILIES.add(family)

def resource_accepts(token, audience):
    claims = jwt.decode(
        token, SIGNING_KEY, algorithms=["HS256"],
        audience=audience, issuer=ISSUER
    )
    if claims.get("delegation_family") in REVOKED_FAMILIES:
        raise PermissionError("delegation revoked")
    return claims

print(resource_accepts(child, "flight-api")["sub"])
revoke_family("dlg:483")

try:
    resource_accepts(child, "flight-api")
except PermissionError as e:
    print("DENIED AFTER REVOCATION:", e)


## 16 — OBO policy model

In [ ]:
def obo_decision(*, agent, requested_scope, consented, delegated_permissions):
    if not consented:
        return False, "user consent missing"
    if requested_scope not in delegated_permissions.get(agent, set()):
        return False, "agent lacks delegated permission"
    return True, "OBO permitted"

delegated_permissions = {
    "agent:travel-supervisor": {"calendar:read","travel:book"}
}

print(obo_decision(
    agent="agent:travel-supervisor",
    requested_scope="calendar:read",
    consented=True,
    delegated_permissions=delegated_permissions,
))


## 17 — Audit exchange evidence

In [ ]:
print(json.dumps(EXCHANGE_AUDIT, indent=2))


## 18 — Exercise: resource attenuation

Implement a hierarchy-aware rule:

```text
parent = project:atlas
child  = project:atlas/document:17 -> allow

parent = project:atlas
child  = tenant:all -> deny
```

Add the resource constraint to the token and verify it at the API.


## 19 — Exercise: approval-bound issuance

Require an approval object before issuing a payment token:

```text
approval_id
actor
action
resource
amount
approver
expires_at
```

The token should contain:

```text
aud = payment-api
scope = payment:create
max_amount = approved amount
TTL <= 60s
approval_id
```


## 20 — Exercise: workload-backed actor

Require both:

```text
logical actor = agent:travel-supervisor
workload = spiffe://corp.example/prod/agent/travel-supervisor
```

Check the agent registry before token exchange.


## 21 — Exercise: sender-constrained token

Add a `cnf` claim representing a DPoP or mTLS key.

Demonstrate:

```text
stolen token only -> deny
token + matching key proof -> allow
```


## 22 — Exercise: cross-domain delegation

Trust domains:

```text
corp.example
vendor.example
```

Permit vendor agent only:

```text
aud = public-search-api
scope = public:search
```

Deny internal documents and payment scopes.


## 23 — Review questions

1. What does RFC 8693 standardize?
2. What does `subject_token` represent?
3. What does `actor_token` represent?
4. What is the difference between `act` and `may_act`?
5. How is delegation different from impersonation?
6. Why are nested `act` claims useful?
7. Which actor should drive current actor-based authorization?
8. Why should historical actors not confer current privilege?
9. What is authority attenuation?
10. What dimensions should be attenuated?
11. Why should output lifetime be bounded by source lifetime?
12. Why should redelegation be explicit?
13. How can actor substitution occur?
14. Why is token type validation necessary?
15. Why does RFC 8693 not automatically solve revocation propagation?
16. What is a delegation family?
17. Why is the token broker a security control plane?
18. How does workload identity improve actor authentication?
19. How do sender-constrained tokens reduce replay risk?
20. Why are vendor OBO implementations not necessarily identical to RFC 8693?

# Next course

## Intermediate 04 — Fine-Grained Authorization with OPA, Cedar & OpenFGA
